In [1]:
%reset -f

In [11]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [12]:
ol = Overlay('resizer-zcu102.bit')

In [7]:
help(ol)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay resizer-zcu102.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_vdma_0           : pynq.lib.video.dma.AxiVDMA
    img2axis_0           : pynq.overlay.DefaultIP
    sensorSupervisor_0   : pynq.overlay.DefaultIP
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [8]:
img2axis = ol.img2axis_0

In [10]:
help(img2axis.register_map)

Help on RegisterMapimg2axis_0 in module pynq.registers object:

class RegisterMapimg2axis_0(RegisterMap)
 |  RegisterMapimg2axis_0(buffer)
 |  
 |  Method resolution order:
 |      RegisterMapimg2axis_0
 |      RegisterMap
 |      builtins.object
 |  
 |  Data descriptors defined here:
 |  
 |  CTRL
 |      Control signals
 |  
 |  GIER
 |      Global Interrupt Enable Register
 |  
 |  IP_IER
 |      IP Interrupt Enable Register
 |  
 |  IP_ISR
 |      IP Interrupt Status Register
 |  
 |  data_port
 |      Data signal of data_port
 |  
 |  end_of_stream
 |      Data signal of end_of_stream
 |  
 |  frame_no
 |      Data signal of frame_no
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from RegisterMap:
 |  
 |  __init__(self, buffer)
 |      Create a new instance of the RegisterMap
 |      
 |      Parameters
 |      ----------
 |      buffer : buffer-like
 |          A Python buffer object to bind the register map to
 |  
 |  __r

In [21]:
def img_to_axis(ip,buffer, eos,frame_no):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.end_of_stream = eos

    # Set frame_no to 88
    ip.frame_no = frame_no
    # Start the IP core by setting the ap_start bit in CTRL register
    ip.register_map.ap_start = 1

In [15]:
def vdma_write(vdma,dst):
    vdma.writechannel.start()    # Enable the write channel
    vdma.writechannel.write(buffer)  # Queue your buffer as destination

In [13]:


def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (r << 16) | (g << 8) | b  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

In [16]:
img_fname='1920x1080-full-hd-nature-landscape.jpg'

In [17]:
buff_o=image_to_RGB(img_fname)

Packed buffer shape: (1080, 1920), dtype: uint32


In [18]:
 buff_i = allocate(shape=buff_o.shape, dtype=np.uint32)

In [22]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [24]:
vdma_write(ol.axi_vdma_0,buff_i)

AttributeError: 'AxiVDMA' object has no attribute 's2mm_introut'